# WasteVision: training notebook
**Explainable CNN for household waste segregation under India's SWM Rules, 2026**

This notebook does everything needed for the model, start to finish:

1. Downloads three public datasets (Garbage Dataset GD v2, TrashBox, DWSD from Kolkata).
2. Cleans them, removes duplicates across sources, maps them to 11 material classes, and makes frozen splits.
3. Trains three models on the same data: a small CNN from scratch, MobileNetV2 and EfficientNetV2B0.
4. Calibrates them (temperature scaling) and evaluates stream accuracy, Special Care recall, risk–coverage,
   per-source accuracy and the Indian held-out set with 95% bootstrap confidence intervals.
5. Draws Grad-CAM heatmaps for correct and wrong predictions.
6. Exports the best model and its settings for the Gradio app.
7. (Later) runs one gated retraining round on feedback collected through the app.

**How to run:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. Allow Google Drive access when asked.
A full run takes roughly 1.5–2.5 hours. Everything important is saved to `MyDrive/WasteVision`, so if Colab
disconnects, just **Run all** again: finished steps are skipped and training resumes with the next model.

**When it finishes:** download `MyDrive/WasteVision/results.zip` for the report and paper.

In [ ]:
#@title 1 · Settings (the only cell you may want to edit)
QUICK_TEST = False   # True = small trial run (~10–15 min) that checks the whole pipeline end to end
RETRAIN = False      # True = train again even if a saved model already exists on Drive

PROJECT_DIR = "/content/drive/MyDrive/WasteVision"   # durable outputs (survive disconnects)
WORK_DIR = "/content/wv_work"                          # fast local scratch space

MODELS = ["scratch_cnn", "mobilenetv2", "efficientnetv2b0"]
IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42

# Data
CAP_PER_CLASS = 1200          # max images per class from GD + TrashBox (DWSD is never capped)
PHASH_DUP_THRESHOLD = 5       # Hamming distance at or below which two web images count as duplicates
DWSD_GROUP_THRESHOLD = 10     # looser threshold to group DWSD photos that are flips/rotations of each other
DWSD_TEST_FRACTION = 0.30     # share of DWSD photo groups held out as the Indian test set
MIN_CROP_SIDE = 40            # DWSD object crops smaller than this (pixels) are dropped
MIN_CROP_AREA = 600

# Training
EPOCHS_SCRATCH = 30
EPOCHS_HEAD = 10              # phase 1: backbone frozen
EPOCHS_FINETUNE = 12          # phase 2: last layers unfrozen
UNFREEZE_LAST_LAYERS = 30
LR_SCRATCH, LR_HEAD, LR_FINETUNE = 1e-3, 1e-3, 1e-5
PRETRAINED_WEIGHTS = "imagenet"

# Decisions
COST_RATIO = 4.0                          # missing a hazardous item is 4x worse than a false alarm -> threshold 1/(1+4) = 0.20
SENSITIVITY_COST_RATIOS = [2.0, 4.0, 9.0] # thresholds 0.33, 0.20, 0.10
TARGET_STREAM_ACC = 0.97                  # "not sure" threshold: keep stream accuracy on answered items at or above this
TARGET_CLASS_ACC = 0.95
BOOTSTRAP_ROUNDS = 1000

In [ ]:
#@title 2 · Setup
import os, sys, json, time, shutil, subprocess, hashlib, re, math, random, glob, zipfile, tarfile, warnings, datetime
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

IN_COLAB = "google.colab" in sys.modules

# Environment overrides (used only for local testing of this notebook)
if os.environ.get("WV_PROJECT_DIR"): PROJECT_DIR = os.environ["WV_PROJECT_DIR"]
if os.environ.get("WV_WORK_DIR"): WORK_DIR = os.environ["WV_WORK_DIR"]
if os.environ.get("WV_QUICK"): QUICK_TEST = os.environ["WV_QUICK"] == "1"
if os.environ.get("WV_WEIGHTS"):
    PRETRAINED_WEIGHTS = None if os.environ["WV_WEIGHTS"].lower() == "none" else os.environ["WV_WEIGHTS"]
LOCAL_SOURCES = json.loads(os.environ.get("WV_LOCAL_SOURCES", "{}"))

def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

try:
    import imagehash
except ImportError:
    pip_install("imagehash")
    import imagehash

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

if QUICK_TEST:
    PROJECT_DIR = PROJECT_DIR.rstrip("/") + "/quick_test"
    WORK_DIR = WORK_DIR.rstrip("/") + "_quick"
    EPOCHS_SCRATCH, EPOCHS_HEAD, EPOCHS_FINETUNE = 2, 1, 1
    BOOTSTRAP_ROUNDS = 200

DATA_DIR = f"{PROJECT_DIR}/data"
MODELS_DIR = f"{PROJECT_DIR}/models"
RESULTS_DIR = f"{PROJECT_DIR}/results"
FIG_DIR = f"{RESULTS_DIR}/figures"
EXPORT_DIR = f"{PROJECT_DIR}/export"
FEEDBACK_DIR = f"{PROJECT_DIR}/feedback"
RAW_DIR = f"{WORK_DIR}/raw"
NORM_DIR = f"{WORK_DIR}/norm"
CACHE_DIR = f"{WORK_DIR}/tfcache"
for d in [DATA_DIR, MODELS_DIR, RESULTS_DIR, FIG_DIR, EXPORT_DIR, f"{FEEDBACK_DIR}/images", RAW_DIR, NORM_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*shuffle=True.*")
warnings.filterwarnings("ignore", message=".*least populated class.*")
random.seed(SEED); np.random.seed(SEED)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)

# ---- The 11 material classes and their SWM Rules 2026 streams ----
CLASSES = ["biological", "paper_cardboard", "plastic", "metal", "glass", "clothes", "shoes",
           "trash", "battery", "medicines_syringes", "e_waste"]
CLASS_DISPLAY = {
    "biological": "Food & organic", "paper_cardboard": "Paper & cardboard", "plastic": "Plastic (incl. thermocol)",
    "metal": "Metal", "glass": "Glass", "clothes": "Clothes & textiles", "shoes": "Shoes",
    "trash": "Other dry residue", "battery": "Battery", "medicines_syringes": "Medicines & syringes",
    "e_waste": "E-waste",
}
STREAMS = ["Wet", "Dry", "Special Care"]     # Sanitary is not modelled (no usable public image data)
STREAM_OF = {c: "Dry" for c in CLASSES}
STREAM_OF["biological"] = "Wet"
for c in ["battery", "medicines_syringes", "e_waste"]:
    STREAM_OF[c] = "Special Care"
NUM_CLASSES = len(CLASSES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
CLASS_STREAM_IDX = np.array([STREAMS.index(STREAM_OF[c]) for c in CLASSES])
STREAM_MATRIX = np.zeros((NUM_CLASSES, len(STREAMS)), np.float32)
STREAM_MATRIX[np.arange(NUM_CLASSES), CLASS_STREAM_IDX] = 1.0
SC = STREAMS.index("Special Care")
SC_THRESHOLD = 1.0 / (1.0 + COST_RATIO)

STATS_PATH = f"{RESULTS_DIR}/stats.json"
STATS = json.load(open(STATS_PATH)) if os.path.exists(STATS_PATH) else {}
def save_stats():
    with open(STATS_PATH, "w") as f:
        json.dump(STATS, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))

def stable_id(*parts):
    return hashlib.sha1("|".join(str(p) for p in parts).encode()).hexdigest()[:16]

print(f"Colab: {IN_COLAB} | quick test: {QUICK_TEST} | project folder: {PROJECT_DIR}")

## Step 1 · Get the data
If the prepared images and frozen splits already exist on Drive (from an earlier run), they are restored and
the download and cleaning cells are skipped automatically. The test sets therefore stay identical across runs.

In [ ]:
#@title 3 · Restore prepared data if it exists
SPLITS_CSV = f"{DATA_DIR}/splits.csv"
NORM_TAR = f"{DATA_DIR}/normalized_images.tar"

def safe_extract_tar(tar_path, dest):
    with tarfile.open(tar_path) as t:
        if hasattr(tarfile, "data_filter"):
            t.extractall(dest, filter="data")
        else:
            for m in t.getmembers():
                if m.name.startswith("/") or ".." in Path(m.name).parts:
                    raise RuntimeError(f"Unsafe path in archive: {m.name}")
            t.extractall(dest)

DATA_READY = False
if os.path.exists(SPLITS_CSV) and os.path.exists(NORM_TAR):
    if not any(Path(NORM_DIR).rglob("*.jpg")):
        print("Restoring prepared images from Drive ...")
        safe_extract_tar(NORM_TAR, NORM_DIR)
    DATA_READY = True
    print(f"✓ Prepared data found: {SPLITS_CSV}. Skipping download and cleaning.")
else:
    print("No prepared data yet. The next cells will download and prepare it.")

In [ ]:
#@title 4 · Download the three datasets
GD_SLUG = "sumn2u/garbage-classification-v2"
TRASHBOX_REPO = "https://github.com/nikhilvenkatkumsetty/TrashBox"
TRASHBOX_SUBDIR = "TrashBox_train_dataset_subfolders"
DWSD_URL = ("https://data.mendeley.com/public-files/datasets/gr99ny6b8p/files/"
            "d0a8df4f-a6d5-447a-95f9-882c7b8fb896/file_downloaded")
DWSD_SHA256 = "fb9229c330ed253f4fb7c18bd46dddebe4df6f8d3f602747f505ff7601a11737"

def run(cmd):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)

def find_dir(root, name):
    for r, ds, _ in os.walk(root):
        for d in ds:
            if d.lower() == name.lower():
                return os.path.join(r, d)
    return None

def get_gd():
    if LOCAL_SOURCES.get("gd"):
        return LOCAL_SOURCES["gd"]
    try:
        import kagglehub
    except ImportError:
        pip_install("kagglehub"); import kagglehub
    if IN_COLAB:  # optional: Kaggle credentials stored as Colab secrets
        try:
            from google.colab import userdata
            for k in ("KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"):
                try:
                    os.environ.setdefault(k, userdata.get(k))
                except Exception:
                    pass
        except Exception:
            pass
    path = kagglehub.dataset_download(GD_SLUG)
    d = find_dir(path, "standardized_256") or find_dir(path, "original")
    if d is None:
        raise RuntimeError(f"Could not find the image folders inside {path}")
    return d

def get_trashbox():
    if LOCAL_SOURCES.get("trashbox"):
        return LOCAL_SOURCES["trashbox"]
    dest = f"{RAW_DIR}/trashbox"
    target = f"{dest}/{TRASHBOX_SUBDIR}"
    if not os.path.isdir(target) or sum(1 for _ in Path(target).rglob("*.jpg")) < 1000:
        if os.path.isdir(dest):
            shutil.rmtree(dest)
        # Partial, sparse clone: downloads only the training images, not the whole repository history
        run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", TRASHBOX_REPO, dest])
        run(["git", "-C", dest, "sparse-checkout", "set", TRASHBOX_SUBDIR])
    return target

def download(url, dst):
    import urllib.request
    tmp = dst + ".part"
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=120) as r, open(tmp, "wb") as f:
        shutil.copyfileobj(r, f, length=1 << 20)
    os.replace(tmp, dst)

def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def safe_extract_zip(zip_path, dest):
    with zipfile.ZipFile(zip_path) as z:
        for m in z.namelist():
            p = Path(m)
            if p.is_absolute() or ".." in p.parts:
                raise RuntimeError(f"Unsafe path in zip: {m}")
        z.extractall(dest)

def get_dwsd():
    zip_path = LOCAL_SOURCES.get("dwsd_zip") or f"{RAW_DIR}/DSWD.zip"
    manual = f"{PROJECT_DIR}/raw/DSWD.zip"   # fallback: upload the zip here by hand if the download fails
    if not os.path.exists(zip_path) and os.path.exists(manual):
        zip_path = manual
    if not os.path.exists(zip_path):
        print("Downloading DWSD (about 250 MB) ...")
        download(DWSD_URL, zip_path)
    STATS["dwsd_zip_sha256_matches"] = sha256_of(zip_path) == DWSD_SHA256
    out = f"{RAW_DIR}/dwsd"
    if not os.path.isdir(out) or not any(Path(out).rglob("*.*")):
        os.makedirs(out, exist_ok=True)
        safe_extract_zip(zip_path, out)
        for inner in list(Path(out).rglob("*.zip")):   # some releases nest a second zip
            safe_extract_zip(str(inner), str(inner.parent / inner.stem))
    return out

SOURCES = {}
if not DATA_READY:
    for name, fn in [("gd", get_gd), ("trashbox", get_trashbox), ("dwsd", get_dwsd)]:
        try:
            SOURCES[name] = fn()
            print(f"✓ {name}: {SOURCES[name]}")
        except Exception as e:
            print(f"✗ {name} failed: {type(e).__name__}: {e}")
            if name != "dwsd":
                raise
            print("  Continuing WITHOUT DWSD: training still works, but the Indian test set is skipped.\n"
                  "  To fix: download DSWD.zip from https://data.mendeley.com/datasets/gr99ny6b8p/1\n"
                  f"  and upload it to {PROJECT_DIR}/raw/DSWD.zip, then run all cells again.")

In [ ]:
#@title 5 · Map GD and TrashBox folders to the 11 classes
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".gif", ".tif", ".tiff"}

def list_images(root):
    return sorted(str(p) for p in Path(root).rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS)

GD_MAP = {"battery": "battery", "biological": "biological", "cardboard": "paper_cardboard", "clothes": "clothes",
          "glass": "glass", "metal": "metal", "paper": "paper_cardboard", "plastic": "plastic",
          "shoes": "shoes", "trash": "trash"}

def trashbox_label(top, sub):
    """TrashBox folder -> our class. None = excluded (no clean home in the SWM 2026 streams)."""
    top, sub = top.lower().strip(), (sub or "").lower().strip()
    if top == "cardboard" or top == "paper":
        return "paper_cardboard"
    if top == "glass":
        return "glass"
    if top == "e-waste":
        return "e_waste"
    if top == "plastic":
        return None if "cigarette" in sub else "plastic"
    if top == "metal":
        return None if "construction" in sub else "metal"
    if top == "medical":
        return "medicines_syringes" if sub in ("medicines", "syringe", "syringes") else None  # gloves, masks excluded
    return None

def build_web_manifest():
    rows = []
    gd = SOURCES["gd"]
    for p in list_images(gd):
        rel = os.path.relpath(p, gd)
        cls = rel.split(os.sep)[0]
        rows.append(dict(source="GD", src_path=p, rel=rel, orig_class=cls, label=GD_MAP.get(cls.lower())))
    tb = SOURCES["trashbox"]
    for p in list_images(tb):
        rel = os.path.relpath(p, tb)
        parts = rel.split(os.sep)
        top = parts[0]
        sub = parts[1] if len(parts) > 2 else None
        rows.append(dict(source="TrashBox", src_path=p, rel=rel,
                         orig_class=f"{top}/{sub}" if sub else top, label=trashbox_label(top, sub)))
    df = pd.DataFrame(rows)
    df["id"] = [stable_id(s, r) for s, r in zip(df.source, df.rel)]
    df["bbox"] = None
    df["group"] = df["id"]
    excluded = df[df.label.isna()].groupby(["source", "orig_class"]).size()
    STATS["excluded_folders"] = {f"{a}:{b}": int(n) for (a, b), n in excluded.items()}
    STATS["raw_counts"] = {k: int(v) for k, v in df.groupby("source").size().items()}
    df = df[df.label.notna()].copy()
    if QUICK_TEST:
        df = df.sample(frac=1, random_state=SEED).groupby(["source", "orig_class"]).head(8)
    return df.reset_index(drop=True)

if not DATA_READY:
    df_web = build_web_manifest()
    print("Images per source after mapping:", df_web.groupby("source").size().to_dict())
    print("Excluded folders:", STATS.get("excluded_folders"))
    display(df_web.groupby(["label", "source"]).size().unstack(fill_value=0))

In [ ]:
#@title 6 · DWSD: read masks, cut out one crop per object, group near-identical photos
DWSD_CLASSES = {1: "plastic-container", 2: "plastic-bottle", 3: "thermocol", 4: "metal-bottle",
                5: "plastic-cardboard", 6: "glass", 7: "thermocol-plate", 8: "plastic", 9: "paper",
                10: "plastic-cup", 11: "paper-cup", 12: "aluminium-foil", 13: "cloth", 14: "nylon"}
DWSD_TO_OURS = {"plastic-container": "plastic", "plastic-bottle": "plastic", "thermocol": "plastic",
                "metal-bottle": "metal", "plastic-cardboard": None, "glass": "glass", "thermocol-plate": "plastic",
                "plastic": "plastic", "paper": "paper_cardboard", "plastic-cup": "plastic",
                "paper-cup": "paper_cardboard", "aluminium-foil": "metal", "cloth": "clothes", "nylon": "plastic"}
# Fallback only: class ids ordered by total pixel count in the DWSD paper (Table 3), largest first
DWSD_PIXEL_RANK = [3, 5, 12, 10, 8, 11, 4, 6, 9, 13, 1, 14, 2, 7]
MASK_WORDS = ("mask", "label", "annot", "gt", "segment", "ground")

POP8 = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
def popcount64(x):
    if hasattr(np, "bitwise_count"):
        return np.bitwise_count(x)
    return POP8[x.view(np.uint8).reshape(x.shape + (8,))].sum(-1)

class UnionFind:
    def __init__(self, n): self.p = list(range(n))
    def find(self, a):
        while self.p[a] != a:
            self.p[a] = self.p[self.p[a]]; a = self.p[a]
        return a
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.p[rb] = ra

def voc_colormap(n=256):
    cmap = np.zeros((n, 3), np.uint8)
    for i in range(n):
        r = g = b = 0; c = i
        for j in range(8):
            r |= ((c >> 0) & 1) << (7 - j); g |= ((c >> 1) & 1) << (7 - j); b |= ((c >> 2) & 1) << (7 - j)
            c >>= 3
        cmap[i] = (r, g, b)
    return cmap
VOC_CODES = {int(r) << 16 | int(g) << 8 | int(b): i for i, (r, g, b) in enumerate(voc_colormap(16))}

def color_codes(rgb):
    return (rgb[..., 0].astype(np.int64) << 16) | (rgb[..., 1].astype(np.int64) << 8) | rgb[..., 2].astype(np.int64)

def read_mask(path):
    """Returns ('index', array of class ids) or ('color', RGB array) or ('bad', None)."""
    m = Image.open(path)
    if m.mode == "P":
        return "index", np.array(m).astype(np.int64)
    if m.mode in ("L", "I", "I;16", "I;16B", "F", "1"):
        arr = np.array(m).astype(np.int64)
    else:
        rgb = np.array(m.convert("RGB"))
        if np.array_equal(rgb[..., 0], rgb[..., 1]) and np.array_equal(rgb[..., 1], rgb[..., 2]):
            arr = rgb[..., 0].astype(np.int64)
        else:
            return "color", rgb
    if arr.max() > 15:   # masks saved scaled up for viewing (e.g. id*17); the scale is found across all masks
        return "scaled", arr
    return "index", arr

def norm_key(stem):
    s = stem.lower()
    for w in ("mask", "image", "img", "label", "annotation", "annot", "groundtruth", "ground_truth", "segmentation", "_gt", "gt_"):
        s = s.replace(w, "")
    return re.sub(r"[^a-z0-9]", "", s)

def split_hint(rel_parts):
    for p in rel_parts:
        q = p.lower()
        for h in ("train", "test", "val"):
            if h in q:
                return h
    return ""

def phash_int(img):
    return int(str(imagehash.phash(img)), 16)

def dihedral_hashes(img):
    hs = []
    for k in range(4):
        r = img.rotate(90 * k, expand=True)
        hs += [phash_int(r), phash_int(ImageOps.mirror(r))]
    return hs

def build_dwsd_manifest(root):
    files = list_images(root)
    masks, images = {}, {}
    for p in files:
        rel = Path(os.path.relpath(p, root))
        is_mask = any(w in part.lower() for part in rel.parts for w in MASK_WORDS)
        key = (split_hint(rel.parts[:-1]), norm_key(rel.stem))
        (masks if is_mask else images)[key] = p
    pairs = [(images[k], masks[k]) for k in images if k in masks]
    STATS["dwsd_files"] = dict(images=len(images), masks=len(masks), paired=len(pairs))
    print(f"DWSD: {len(images)} images, {len(masks)} masks, {len(pairs)} paired")
    if not pairs:
        print("Could not pair DWSD images with masks. Folder sample:", [os.path.relpath(f, root) for f in files[:15]])
        return None

    # 1) decode all masks to class-id arrays
    decoded, color_masks, scaled, kinds = {}, {}, {}, []
    for img_p, m_p in pairs:
        kind, arr = read_mask(m_p)
        kinds.append(kind)
        if kind == "index": decoded[img_p] = arr
        elif kind == "color": color_masks[img_p] = arr
        elif kind == "scaled": scaled[img_p] = arr
    if scaled:
        vals = np.unique(np.concatenate([np.unique(a) for a in scaled.values()]))
        nz = vals[vals > 0]
        g = int(np.gcd.reduce(nz)) if len(nz) else 1
        if g > 1 and vals.max() // g <= 15:
            for img_p, a in scaled.items():
                decoded[img_p] = a // g
            STATS["dwsd_mask_mapping"] = f"greyscale scaled by {g}"
        else:
            print(f"⚠ {len(scaled)} DWSD masks have values above 15 with no common scale; they are skipped.")
    if color_masks:
        known = total = 0
        for rgb in color_masks.values():
            codes = color_codes(rgb); nz = codes != 0
            total += int(nz.sum()); known += int(np.isin(codes[nz], list(VOC_CODES)).sum())
        if total and known / total >= 0.9:
            mapping, how = VOC_CODES, "voc_palette"
        else:   # unknown palette: match colours to classes by pixel-count rank (flagged in stats)
            counts = {}
            for rgb in color_masks.values():
                u, c = np.unique(color_codes(rgb), return_counts=True)
                for a, b in zip(u, c):
                    if a != 0: counts[int(a)] = counts.get(int(a), 0) + int(b)
            top = sorted(counts, key=counts.get, reverse=True)[:len(DWSD_PIXEL_RANK)]
            mapping, how = {code: cid for code, cid in zip(top, DWSD_PIXEL_RANK)}, "inferred_by_pixel_rank"
            print("⚠ DWSD colour palette unknown; classes inferred by pixel-count rank. Check the crops grid below.")
        for img_p, rgb in color_masks.items():
            codes = color_codes(rgb)
            arr = np.zeros(codes.shape, np.int64)
            for code, cid in mapping.items():
                arr[codes == code] = cid
            decoded[img_p] = arr
        STATS["dwsd_mask_mapping"] = how
    STATS["dwsd_mask_kinds"] = {k: kinds.count(k) for k in set(kinds)}
    print("Mask formats:", STATS["dwsd_mask_kinds"])

    # 2) group parent photos that are flips/rotations/near-copies of each other
    parents = sorted(decoded)
    h8, h0 = [], []
    for p in parents:
        with Image.open(p) as im:
            hs = dihedral_hashes(im.convert("RGB"))
        h8.append(hs); h0.append(hs[0])
    H8, H0 = np.array(h8, dtype=np.uint64), np.array(h0, dtype=np.uint64)
    uf = UnionFind(len(parents))
    for i in range(len(parents)):
        d = popcount64(H8[i][:, None] ^ H0[None, :]).min(0)
        for j in np.nonzero(d <= DWSD_GROUP_THRESHOLD)[0]:
            if j != i: uf.union(i, int(j))
    group_of = {p: f"DWSD_{uf.find(i)}" for i, p in enumerate(parents)}
    STATS["dwsd_parent_photos"] = len(parents)
    STATS["dwsd_parent_groups"] = len(set(group_of.values()))

    # 3) one crop per connected object, padded by 10%
    from scipy import ndimage
    rows, skipped = [], {"too_small": 0, "excluded_class": 0}
    for p in parents:
        arr = decoded[p]
        with Image.open(p) as im:
            W, H = im.size
        if arr.shape != (H, W):
            arr = np.array(Image.fromarray(arr.astype(np.uint8)).resize((W, H), Image.NEAREST)).astype(np.int64)
        rel = os.path.relpath(p, root)
        k = 0
        for cid in np.unique(arr):
            name = DWSD_CLASSES.get(int(cid))
            if name is None:
                continue
            ours = DWSD_TO_OURS[name]
            lab, _ = ndimage.label(arr == cid, structure=np.ones((3, 3)))
            for i, sl in enumerate(ndimage.find_objects(lab), start=1):
                if sl is None: continue
                y0, y1, x0, x1 = sl[0].start, sl[0].stop, sl[1].start, sl[1].stop
                area = int((lab[sl] == i).sum())
                if min(y1 - y0, x1 - x0) < MIN_CROP_SIDE or area < MIN_CROP_AREA:
                    skipped["too_small"] += 1; continue
                if ours is None:
                    skipped["excluded_class"] += 1; continue
                py, px = int(0.1 * (y1 - y0)), int(0.1 * (x1 - x0))
                bbox = (max(0, x0 - px), max(0, y0 - py), min(W, x1 + px), min(H, y1 + py))
                rows.append(dict(source="DWSD", src_path=p, rel=f"{rel}#{k}", orig_class=name, label=ours,
                                 bbox=bbox, group=group_of[p]))
                k += 1
    df = pd.DataFrame(rows)
    STATS["dwsd_crops_skipped"] = skipped
    if df.empty:
        print("No usable DWSD crops found.")
        return None
    df["id"] = [stable_id("DWSD", r) for r in df.rel]
    if QUICK_TEST:
        df = df.sample(frac=1, random_state=SEED).groupby("label").head(12)
    STATS["dwsd_crops_by_class"] = {k: int(v) for k, v in df.groupby("orig_class").size().items()}
    print(f"DWSD crops: {len(df)} from {STATS['dwsd_parent_photos']} photos in {STATS['dwsd_parent_groups']} groups;",
          "skipped:", skipped)
    return df.reset_index(drop=True)

if not DATA_READY:
    df_dwsd = None
    if SOURCES.get("dwsd"):
        try:
            df_dwsd = build_dwsd_manifest(SOURCES["dwsd"])
        except Exception as e:
            print(f"✗ DWSD processing failed ({type(e).__name__}: {e}). Continuing without the Indian set.")
    if df_dwsd is not None:
        display(df_dwsd.groupby(["orig_class", "label"]).size().rename("crops").to_frame())
    save_stats()

In [ ]:
#@title 7 · Clean and standardise every image (RGB JPEG, max side 320 px) and compute perceptual hashes
NORM_MAX_SIDE = 320

def normalize_one(job):
    src, dst, bbox, min_side = job
    try:
        with Image.open(src) as im0:
            im = ImageOps.exif_transpose(im0)
            if im.mode in ("RGBA", "LA") or (im.mode == "P" and "transparency" in im.info):
                im = im.convert("RGBA")
                bg = Image.new("RGB", im.size, (255, 255, 255))
                bg.paste(im, mask=im.getchannel("A"))
                im = bg
            else:
                im = im.convert("RGB")
            if bbox is not None:
                im = im.crop(tuple(int(v) for v in bbox))
            w, h = im.size
            if min(w, h) < min_side:
                return None, "too_small"
            im.thumbnail((NORM_MAX_SIDE, NORM_MAX_SIDE), Image.Resampling.LANCZOS)
            im.save(dst, "JPEG", quality=93)
            return (str(imagehash.phash(im)), w, h), None
    except Exception as e:
        return None, f"unreadable ({type(e).__name__})"

def normalize_all(df):
    jobs, out_rel = [], []
    for r in df.itertuples():
        rel = f"{r.source}/{r.id}.jpg"
        os.makedirs(f"{NORM_DIR}/{r.source}", exist_ok=True)
        out_rel.append(rel)
        jobs.append((r.src_path, f"{NORM_DIR}/{rel}", r.bbox if isinstance(r.bbox, tuple) else None,
                     24 if r.source != "DWSD" else MIN_CROP_SIDE))
    results, t0 = [], time.time()
    with ThreadPoolExecutor(max_workers=min(16, 4 * (os.cpu_count() or 2))) as ex:
        for i, res in enumerate(ex.map(normalize_one, jobs, chunksize=32)):
            results.append(res)
            if (i + 1) % 2000 == 0:
                print(f"  {i + 1}/{len(jobs)} images ({time.time() - t0:.0f}s)")
    df = df.copy()
    df["norm_rel"] = out_rel
    df["phash"] = [r[0][0] if r[0] else None for r in results]
    df["src_w"] = [r[0][1] if r[0] else None for r in results]
    df["src_h"] = [r[0][2] if r[0] else None for r in results]
    fails = pd.Series([r[1] for r in results if r[1]]).value_counts().to_dict()
    STATS["cleaning_dropped"] = {k: int(v) for k, v in fails.items()}
    print(f"Cleaned {len(df)} images in {time.time() - t0:.0f}s; dropped: {fails}")
    return df[df.phash.notna()].reset_index(drop=True)

if not DATA_READY:
    parts = [df_web] + ([df_dwsd] if df_dwsd is not None else [])
    df_all = normalize_all(pd.concat(parts, ignore_index=True))

In [ ]:
#@title 8 · Remove duplicates across GD and TrashBox, cap classes, make frozen splits
from sklearn.model_selection import train_test_split, StratifiedGroupKFold, GroupShuffleSplit

def near_duplicate_pairs(hashes, thr, chunk=512):
    H = np.array(hashes, dtype=np.uint64)
    out = []
    for s in range(0, len(H), chunk):
        d = popcount64(H[s:s + chunk, None] ^ H[None, :])
        ii, jj = np.nonzero(d <= thr)
        ii = ii + s
        keep = jj > ii
        out.append(np.stack([ii[keep], jj[keep], d[ii[keep] - s, jj[keep]]], 1))
    return np.concatenate(out) if out else np.zeros((0, 3), np.int64)

def dedup_web(df):
    hashes = [int(h, 16) for h in df.phash]
    pairs = near_duplicate_pairs(hashes, PHASH_DUP_THRESHOLD)
    uf = UnionFind(len(df))
    for i, j, _ in pairs:
        uf.union(int(i), int(j))
    root = np.array([uf.find(i) for i in range(len(df))])
    df = df.assign(cluster=root)
    priority = {"TrashBox": 0, "GD": 1}
    keep, conflicting_clusters = [], 0
    for _, g in df.groupby("cluster"):
        if len(g) == 1:
            keep.append(g.index[0]); continue
        if g.label.nunique() > 1:      # same picture with different labels: ambiguous, drop all copies
            conflicting_clusters += 1; continue
        keep.append(g.assign(pr=g.source.map(priority)).sort_values("pr").index[0])
    src = df.source.values
    STATS["dedup"] = dict(
        threshold=PHASH_DUP_THRESHOLD, images_before=len(df), images_after=len(keep),
        duplicate_pairs=len(pairs), exact_pairs=int((pairs[:, 2] == 0).sum()) if len(pairs) else 0,
        cross_source_pairs=int(sum(src[i] != src[j] for i, j, _ in pairs)),
        clusters_with_conflicting_labels=conflicting_clusters)
    print("Deduplication:", STATS["dedup"])
    return df.loc[keep].drop(columns="cluster").reset_index(drop=True)

def strat_split(df, test_size, seed=SEED):
    for key in ("strat", "label", None):
        try:
            return train_test_split(df, test_size=test_size, random_state=seed,
                                    stratify=df[key] if key else None)
        except ValueError:
            continue

def make_splits(df_all):
    web = dedup_web(df_all[df_all.source != "DWSD"])
    web = web.sample(frac=1, random_state=SEED).groupby("label").head(CAP_PER_CLASS)
    web["strat"] = web.label + "|" + web.source
    tr, rest = strat_split(web, 0.30)
    va, te = strat_split(rest, 0.50)
    tr, va, te = tr.assign(split="train"), va.assign(split="val"), te.assign(split="test")
    frames = [tr, va, te]
    dw = df_all[df_all.source == "DWSD"].copy()
    if len(dw):
        n_groups = dw.group.nunique()
        if n_groups >= 10:
            folds = np.zeros(len(dw), int)
            sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=SEED)
            for f, (_, idx) in enumerate(sgkf.split(dw, dw.label, dw.group)):
                folds[idx] = f
            test_mask = folds < round(10 * DWSD_TEST_FRACTION)
        else:
            gss = GroupShuffleSplit(n_splits=1, test_size=DWSD_TEST_FRACTION, random_state=SEED)
            _, idx = next(gss.split(dw, dw.label, dw.group))
            test_mask = np.zeros(len(dw), bool); test_mask[idx] = True
        dw["split"] = np.where(test_mask, "dwsd_test", "train")
        frames.append(dw)
    out = pd.concat(frames, ignore_index=True)
    out["y"] = out.label.map(CLASS_TO_IDX).astype(int)
    cols = ["id", "source", "orig_class", "label", "y", "split", "group", "norm_rel", "phash", "src_w", "src_h"]
    return out[cols]

def leakage_audit(df):
    """Near-duplicates (pHash distance <= threshold) between train and each held-out split."""
    res = {}
    tr = df[df.split == "train"]
    H_tr = np.array([int(h, 16) for h in tr.phash], dtype=np.uint64)
    for s in ["val", "test", "dwsd_test"]:
        ho = df[df.split == s]
        if not len(ho):
            continue
        H = np.array([int(h, 16) for h in ho.phash], dtype=np.uint64)
        n = 0
        for a in range(0, len(H), 256):
            n += int((popcount64(H[a:a + 256, None] ^ H_tr[None, :]).min(1) <= PHASH_DUP_THRESHOLD).sum())
        res[s] = n
    STATS["leakage_audit_near_duplicates_vs_train"] = res
    print("Held-out images with a near-duplicate in train:", res)

if not DATA_READY:
    df_split = make_splits(df_all)
    leakage_audit(df_split)
    df_split.to_csv(SPLITS_CSV, index=False)
    print("Saving prepared images to Drive (one-time) ...")
    with tarfile.open(NORM_TAR + ".part", "w") as t:
        for rel in df_split.norm_rel:
            t.add(f"{NORM_DIR}/{rel}", arcname=rel)
    os.replace(NORM_TAR + ".part", NORM_TAR)
    save_stats()
    print(f"✓ Frozen splits saved to {SPLITS_CSV}")

In [ ]:
#@title 9 · Data summary (tables and figures for the report)
df_split = pd.read_csv(SPLITS_CSV)
df_split["path"] = NORM_DIR + "/" + df_split.norm_rel
missing = (~df_split.path.map(os.path.exists)).sum()
assert missing == 0, f"{missing} prepared images are missing; delete {NORM_TAR} and {SPLITS_CSV} and rerun."

table = df_split.pivot_table(index="label", columns=["split", "source"], values="id", aggfunc="count", fill_value=0)
table = table.reindex(CLASSES)
table.to_csv(f"{RESULTS_DIR}/data_counts.csv")
display(table)
STATS["data_counts"] = {s: {k: int(v) for k, v in g.groupby("label").size().items()}
                        for s, g in df_split.groupby("split")}
STATS["data_totals"] = {s: int(n) for s, n in df_split.groupby("split").size().items()}
STATS["data_by_source"] = {s: int(n) for s, n in df_split.groupby("source").size().items()}
save_stats()

fig, ax = plt.subplots(figsize=(11, 4))
cnt = df_split.groupby(["label", "source"]).size().unstack(fill_value=0).reindex(CLASSES)
cnt.plot(kind="bar", stacked=True, ax=ax, color={"GD": "#4C72B0", "TrashBox": "#DD8452", "DWSD": "#55A868"})
ax.set_xlabel(""); ax.set_ylabel("Images"); ax.set_title("Images per class and source (after cleaning, deduplication and capping)")
ax.set_xticklabels([CLASS_DISPLAY[c] for c in CLASSES], rotation=30, ha="right")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/data_distribution.png", dpi=200); plt.show()

def sample_grid(frame, title, path, per_class=3):
    cls = [c for c in CLASSES if (frame.label == c).any()]
    fig, axes = plt.subplots(len(cls), per_class, figsize=(per_class * 2.1, len(cls) * 2.1))
    axes = np.atleast_2d(axes)
    for r, c in enumerate(cls):
        g = frame[frame.label == c]
        picks = g.assign(k=g.groupby("source").cumcount()).sort_values("k", kind="stable").head(per_class)  # mix sources
        for k in range(per_class):
            a = axes[r, k]; a.axis("off")
            if k < len(picks):
                row = picks.iloc[k]
                a.imshow(Image.open(row.path)); a.set_title(f"{c} · {row.source}", fontsize=7)
    fig.suptitle(title); plt.tight_layout(); plt.savefig(path, dpi=150); plt.show()

sample_grid(df_split.sample(frac=1, random_state=1), "Example training images", f"{FIG_DIR}/data_samples.png")
if (df_split.source == "DWSD").any():
    sample_grid(df_split[df_split.source == "DWSD"].sample(frac=1, random_state=1),
                "DWSD (Kolkata) object crops", f"{FIG_DIR}/data_samples_dwsd.png", per_class=4)

## Step 2 · Train three models
Each finished model is saved to Drive immediately. If the session disconnects, run all cells again and
training continues with the next unfinished model.

In [ ]:
#@title 10 · TensorFlow and input pipeline
import tensorflow as tf
import keras
from keras import layers

GPUS = tf.config.list_physical_devices("GPU")
for g in GPUS:
    try: tf.config.experimental.set_memory_growth(g, True)
    except Exception: pass
STATS["environment"] = dict(tensorflow=tf.__version__, keras=keras.__version__, python=sys.version.split()[0],
                            gpu=[tf.config.experimental.get_device_details(g).get("device_name", "GPU") for g in GPUS] or "none")
print(STATS["environment"])
if not GPUS:
    print("⚠ No GPU found. Use Runtime → Change runtime type → T4 GPU, otherwise training will be very slow.")

for f in glob.glob(f"{CACHE_DIR}/*.lockfile"):
    os.remove(f)

SPLITS = {s: df_split[df_split.split == s].reset_index(drop=True) for s in ["train", "val", "test", "dwsd_test"]}
AUTOTUNE = tf.data.AUTOTUNE

def load_image(path):
    """The single preprocessing used everywhere (training, evaluation, Grad-CAM and the app)."""
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    img = tf.image.resize_with_pad(img, IMG_SIZE, IMG_SIZE, antialias=True)
    return tf.cast(tf.clip_by_value(tf.round(img), 0, 255), tf.uint8)

def make_ds(frame, name, training=False):
    paths = frame.path.values.astype(str)
    ys = frame.y.values.astype("int32")
    ds = tf.data.Dataset.from_tensor_slices((paths, ys))
    ds = ds.map(lambda p, y: (load_image(p), y), num_parallel_calls=AUTOTUNE)
    key = hashlib.md5("".join(frame.id).encode()).hexdigest()[:10]
    ds = ds.cache(f"{CACHE_DIR}/{name}_{key}")
    if training:
        ds = ds.shuffle(min(len(frame), 2048), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(BATCH_SIZE).map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

DS = {s: make_ds(f, s, training=(s == "train")) for s, f in SPLITS.items() if len(f)}
counts = np.bincount(SPLITS["train"].y, minlength=NUM_CLASSES)
CLASS_WEIGHTS = {i: float(np.clip(counts.sum() / (NUM_CLASSES * c), 0.5, 5.0)) if c else 1.0
                 for i, c in enumerate(counts)}
print({s: len(f) for s, f in SPLITS.items()})
print("Class weights:", {CLASSES[i]: round(w, 2) for i, w in CLASS_WEIGHTS.items()})

In [ ]:
#@title 11 · Model definitions
@keras.saving.register_keras_serializable(package="WasteVision")
class ColorJitter(layers.Layer):
    """Per-image random brightness, contrast, saturation and hue (training only).
    Mimics glare, dim rooms, cheap phone cameras and food stains."""
    def __init__(self, brightness=0.25, contrast=0.25, saturation=0.35, hue=0.03, **kwargs):
        super().__init__(**kwargs)
        self.brightness, self.contrast, self.saturation, self.hue = brightness, contrast, saturation, hue

    def call(self, x, training=None):
        if not training:
            return x
        b = tf.shape(x)[0]
        x = tf.clip_by_value(x / 255.0, 0.0, 1.0)
        h, s, v = tf.unstack(tf.image.rgb_to_hsv(x), axis=-1)
        h = tf.math.floormod(h + tf.random.uniform([b, 1, 1], -self.hue, self.hue), 1.0)
        s = tf.clip_by_value(s * tf.random.uniform([b, 1, 1], 1 - self.saturation, 1 + self.saturation), 0.0, 1.0)
        v = tf.clip_by_value(v * tf.random.uniform([b, 1, 1], 1 - self.brightness, 1 + self.brightness), 0.0, 1.0)
        x = tf.image.hsv_to_rgb(tf.stack([h, s, v], axis=-1))
        mean = tf.reduce_mean(x, axis=[1, 2, 3], keepdims=True)
        x = (x - mean) * tf.random.uniform([b, 1, 1, 1], 1 - self.contrast, 1 + self.contrast) + mean
        return tf.clip_by_value(x, 0.0, 1.0) * 255.0

    def compute_output_shape(self, input_shape):
        return input_shape

    def get_config(self):
        return {**super().get_config(), "brightness": self.brightness, "contrast": self.contrast,
                "saturation": self.saturation, "hue": self.hue}

def make_augmenter():
    return keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical", name="aug_flip"),
        layers.RandomRotation(0.08, fill_mode="reflect", name="aug_rotate"),
        layers.RandomZoom((-0.15, 0.15), fill_mode="reflect", name="aug_zoom"),
        layers.RandomTranslation(0.08, 0.08, fill_mode="reflect", name="aug_translate"),
        ColorJitter(name="aug_color"),
    ], name="augment")

def scratch_backbone():
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3), name="scratch_in")
    x = inp
    for b, f in enumerate([32, 64, 128, 192, 256]):
        for k in range(2):
            x = layers.Conv2D(f, 3, padding="same", use_bias=False, name=f"b{b}_conv{k}")(x)
            x = layers.BatchNormalization(name=f"b{b}_bn{k}")(x)
            x = layers.ReLU(name=f"b{b}_relu{k}")(x)
        x = layers.MaxPooling2D(name=f"b{b}_pool")(x)
    return keras.Model(inp, x, name="backbone")

def make_head(feature_shape):
    inp = keras.Input(feature_shape, name="features")
    x = layers.GlobalAveragePooling2D(name="head_gap")(inp)
    x = layers.Dropout(0.2, name="head_drop1")(x)
    x = layers.Dense(128, activation="relu", name="head_dense")(x)
    x = layers.Dropout(0.3, name="head_drop2")(x)
    out = layers.Dense(NUM_CLASSES, name="logits")(x)   # logits; softmax (with temperature) applied at inference
    return keras.Model(inp, out, name="head")

def build_model(arch, weights=None):
    """Returns (training model with augmentation, inference model, backbone).
    Both models share the same preprocessing, backbone and head layers."""
    shape = (IMG_SIZE, IMG_SIZE, 3)
    if arch == "scratch_cnn":
        preproc, backbone, frozen_bn = layers.Rescaling(1 / 255.0, name="preproc"), scratch_backbone(), False
    elif arch == "mobilenetv2":
        preproc = layers.Rescaling(1 / 127.5, offset=-1.0, name="preproc")
        base = keras.applications.MobileNetV2(input_shape=shape, include_top=False, weights=weights)
        backbone, frozen_bn = keras.Model(base.input, base.output, name="backbone"), True
    elif arch == "efficientnetv2b0":
        preproc = layers.Identity(name="preproc")   # EfficientNetV2 scales 0-255 input internally
        base = keras.applications.EfficientNetV2B0(input_shape=shape, include_top=False, weights=weights,
                                                   include_preprocessing=True)
        backbone, frozen_bn = keras.Model(base.input, base.output, name="backbone"), True
    else:
        raise ValueError(arch)
    head = make_head(backbone.output_shape[1:])

    def forward(x):
        x = preproc(x)
        x = backbone(x, training=False) if frozen_bn else backbone(x)
        return head(x)

    inp = keras.Input(shape, name="image")
    train_model = keras.Model(inp, forward(make_augmenter()(inp)), name=f"{arch}_train")
    inp2 = keras.Input(shape, name="image")
    infer_model = keras.Model(inp2, forward(inp2), name=arch)
    return train_model, infer_model, backbone

def unfreeze_top(backbone, n):
    backbone.trainable = True
    for l in backbone.layers[:-n]:
        l.trainable = False
    for l in backbone.layers:
        if isinstance(l, layers.BatchNormalization):
            l.trainable = False   # keep pretrained batch-norm statistics fixed while fine-tuning

for arch in MODELS:
    _, m, _ = build_model(arch, weights=None)
    print(f"{arch:18s} parameters: {m.count_params() / 1e6:.2f} M")
keras.backend.clear_session()

In [ ]:
#@title 12 · Train (skips models already saved on Drive)
class KeepBest(keras.callbacks.Callback):
    """Restore the weights from the epoch with the lowest validation loss."""
    def __init__(self):
        super().__init__(); self.best, self.weights = np.inf, None
    def on_epoch_end(self, epoch, logs=None):
        v = (logs or {}).get("val_loss")
        if v is not None and v < self.best:
            self.best, self.weights = v, self.model.get_weights()
    def on_train_end(self, logs=None):
        if self.weights is not None:
            self.model.set_weights(self.weights)

def compile_model(m, lr):
    m.compile(optimizer=keras.optimizers.Adam(lr),
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")], jit_compile=False)

def fit(m, train_ds, epochs, initial_epoch, patience, log_path):
    cbs = [keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience),
           keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=max(1, patience // 2),
                                             min_lr=1e-7, verbose=1),
           KeepBest(), keras.callbacks.CSVLogger(log_path, append=True)]
    return m.fit(train_ds, validation_data=DS["val"], epochs=initial_epoch + epochs, initial_epoch=initial_epoch,
                 class_weight=CLASS_WEIGHTS, callbacks=cbs, verbose=1 if IN_COLAB else 2)

def to_floats(h):
    return {k: [float(v) for v in vals] for k, vals in h.history.items()}

for arch in MODELS:
    out = f"{MODELS_DIR}/{arch}"
    os.makedirs(out, exist_ok=True)
    if os.path.exists(f"{out}/infer.keras") and not RETRAIN:
        print(f"✓ {arch}: already trained, skipping"); continue
    print(f"\n========== Training {arch} ==========")
    keras.utils.set_random_seed(SEED)
    train_m, infer_m, backbone = build_model(arch, weights=None if arch == "scratch_cnn" else PRETRAINED_WEIGHTS)
    log = f"{out}/training_log.csv"
    if os.path.exists(log): os.remove(log)
    t0, phases = time.time(), {}
    if arch == "scratch_cnn":
        compile_model(train_m, LR_SCRATCH)
        phases["full"] = to_floats(fit(train_m, DS["train"], EPOCHS_SCRATCH, 0, 5, log))
    else:
        backbone.trainable = False
        compile_model(train_m, LR_HEAD)
        phases["head"] = to_floats(fit(train_m, DS["train"], EPOCHS_HEAD, 0, 3, log))
        unfreeze_top(backbone, UNFREEZE_LAST_LAYERS)
        compile_model(train_m, LR_FINETUNE)
        phases["finetune"] = to_floats(fit(train_m, DS["train"], EPOCHS_FINETUNE,
                                           len(phases["head"]["loss"]), 4, log))
    infer_m.save(f"{out}/infer.keras")
    info = dict(arch=arch, train_seconds=time.time() - t0, params=int(infer_m.count_params()),
                trainable_params_final=int(sum(np.prod(v.shape) for v in train_m.trainable_weights)),
                phases=phases, pretrained=None if arch == "scratch_cnn" else PRETRAINED_WEIGHTS)
    json.dump(info, open(f"{out}/train_info.json", "w"), indent=1)
    print(f"✓ {arch} trained in {info['train_seconds'] / 60:.1f} min and saved to {out}")
    del train_m, infer_m, backbone
    keras.backend.clear_session()

## Step 3 · Evaluate: calibration, streams, safety rule, uncertainty, Indian test set

In [ ]:
#@title 13 · Evaluation helpers
from scipy.special import softmax, log_softmax
from scipy.optimize import minimize_scalar
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

def fit_temperature(logits, y):
    def nll(log_t):
        return -log_softmax(logits / np.exp(log_t), axis=1)[np.arange(len(y)), y].mean()
    return float(np.exp(minimize_scalar(nll, bounds=(-3, 3), method="bounded").x))

def ece(probs, y, bins=15):
    conf, pred = probs.max(1), probs.argmax(1)
    edges = np.linspace(0, 1, bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            e += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return float(e)

def stream_probs(p):
    return p @ STREAM_MATRIX

def risk_coverage(conf, correct):
    order = np.argsort(-conf)
    c = correct[order].astype(float)
    k = np.arange(1, len(c) + 1)
    return k / len(c), np.cumsum(c) / k, conf[order]

def pick_threshold(conf, correct, target):
    """Lowest confidence threshold whose answered items still reach the target accuracy (max coverage)."""
    cov, acc, th = risk_coverage(conf, correct)
    ok = np.nonzero(acc >= target)[0]
    if not len(ok):
        return float(th[max(0, int(0.8 * len(th)) - 1)]), False
    return float(th[ok.max()]), True

def selective(conf, correct, t):
    m = conf >= t
    return dict(coverage=float(m.mean()), accuracy_answered=float(correct[m].mean()) if m.any() else float("nan"))

def metrics(p, y):
    pred = p.argmax(1)
    sp = stream_probs(p)
    s_true = CLASS_STREAM_IDX[y]
    s_pred = sp.argmax(1)
    is_sc = s_true == SC
    out = dict(n=int(len(y)), class_acc=float(accuracy_score(y, pred)),
               macro_f1=float(f1_score(y, pred, average="macro", labels=np.unique(y), zero_division=0)),
               stream_acc=float((s_pred == s_true).mean()),
               stream_acc_top_class=float((CLASS_STREAM_IDX[pred] == s_true).mean()))
    for r in SENSITIVITY_COST_RATIOS:
        t = 1 / (1 + r)
        flag = sp[:, SC] >= t
        out[f"sc_recall@{t:.2f}"] = float(flag[is_sc].mean()) if is_sc.any() else float("nan")
        out[f"sc_false_alarm@{t:.2f}"] = float(flag[~is_sc].mean()) if (~is_sc).any() else float("nan")
    out["sc_recall_argmax"] = float((s_pred[is_sc] == SC).mean()) if is_sc.any() else float("nan")
    return out

def group_bootstrap_ci(frame, fn, rounds=BOOTSTRAP_ROUNDS):
    """95% interval resampling whole photo groups (crops from one photo stay together)."""
    groups = frame.group.values
    uniq = np.unique(groups)
    idx = {g: np.nonzero(groups == g)[0] for g in uniq}
    rng = np.random.default_rng(SEED)
    vals = []
    for _ in range(rounds):
        ix = np.concatenate([idx[g] for g in rng.choice(uniq, len(uniq), replace=True)])
        vals.append(fn(ix))
    return [float(np.nanpercentile(vals, 2.5)), float(np.nanpercentile(vals, 97.5))]

def measure_latency(model, runs=30):
    x = tf.random.uniform((1, IMG_SIZE, IMG_SIZE, 3), 0, 255)
    res = {}
    for name, dev in ([("gpu", "/GPU:0")] if GPUS else []) + [("cpu", "/CPU:0")]:
        with tf.device(dev):
            f = tf.function(lambda t: model(t, training=False))
            for _ in range(5): f(x).numpy()
            t0 = time.perf_counter()
            for _ in range(runs): f(x).numpy()
            res[f"latency_ms_{name}"] = (time.perf_counter() - t0) / runs * 1000
    return res

In [ ]:
#@title 14 · Evaluate all models and pick the best one (chosen on validation macro-F1, never on test)
RESULTS, PRED = {}, {}
for arch in MODELS:
    path = f"{MODELS_DIR}/{arch}/infer.keras"
    if not os.path.exists(path):
        print(f"⚠ {arch} not trained yet, skipping"); continue
    model = keras.models.load_model(path, compile=False)
    logits = {s: model.predict(DS[s], verbose=0) for s in DS if s != "train"}
    y = {s: SPLITS[s].y.values for s in logits}
    T = fit_temperature(logits["val"], y["val"])
    raw = {s: softmax(l, 1) for s, l in logits.items()}
    cal = {s: softmax(l / T, 1) for s, l in logits.items()}
    r = dict(temperature=T)
    for s in logits:
        r[s] = metrics(cal[s], y[s])
    r["ece_val_before"], r["ece_val_after"] = ece(raw["val"], y["val"]), ece(cal["val"], y["val"])
    r["ece_test_before"], r["ece_test_after"] = ece(raw["test"], y["test"]), ece(cal["test"], y["test"])

    # "Not sure" thresholds chosen on validation, then checked on test
    sv, st = stream_probs(cal["val"]), stream_probs(cal["test"])
    t_stream, ok_s = pick_threshold(sv.max(1), sv.argmax(1) == CLASS_STREAM_IDX[y["val"]], TARGET_STREAM_ACC)
    t_class, ok_c = pick_threshold(cal["val"].max(1), cal["val"].argmax(1) == y["val"], TARGET_CLASS_ACC)
    r["not_sure_stream_threshold"], r["not_sure_class_threshold"] = t_stream, t_class
    r["not_sure_targets_met_on_val"] = dict(stream=ok_s, cls=ok_c)
    r["test_selective_stream"] = selective(st.max(1), st.argmax(1) == CLASS_STREAM_IDX[y["test"]], t_stream)
    r["test_selective_class"] = selective(cal["test"].max(1), cal["test"].argmax(1) == y["test"], t_class)

    # Accuracy per data source (shortcut check)
    te = SPLITS["test"]
    r["test_by_source"] = {src: dict(n=int((te.source == src).sum()),
                                     class_acc=float((cal["test"].argmax(1)[te.source == src] == y["test"][te.source == src]).mean()))
                           for src in te.source.unique()}

    # Indian held-out set with grouped bootstrap intervals
    if "dwsd_test" in cal:
        p, yy, fr = cal["dwsd_test"], y["dwsd_test"], SPLITS["dwsd_test"]
        pred, sp = p.argmax(1), stream_probs(p)
        r["dwsd_ci"] = dict(
            class_acc=group_bootstrap_ci(fr, lambda ix: (pred[ix] == yy[ix]).mean()),
            stream_acc=group_bootstrap_ci(fr, lambda ix: (sp[ix].argmax(1) == CLASS_STREAM_IDX[yy[ix]]).mean()),
            sc_false_alarm=group_bootstrap_ci(fr, lambda ix: (sp[ix, SC] >= SC_THRESHOLD).mean()))
        r["dwsd_groups"] = int(fr.group.nunique())

    info = json.load(open(f"{MODELS_DIR}/{arch}/train_info.json"))
    r.update(params_m=info["params"] / 1e6, size_mb=os.path.getsize(path) / 1e6,
             train_minutes=info["train_seconds"] / 60, **measure_latency(model))
    RESULTS[arch], PRED[arch] = r, dict(raw=raw, cal=cal, y=y)
    print(f"✓ {arch}: test class acc {r['test']['class_acc']:.3f} | stream acc {r['test']['stream_acc']:.3f} | "
          f"T={T:.2f}")
    del model
    keras.backend.clear_session()

json.dump(RESULTS, open(f"{RESULTS_DIR}/results_all_models.json", "w"), indent=1)

def fmt_ci(v, ci):
    return f"{v:.3f} [{ci[0]:.3f}, {ci[1]:.3f}]"

t20 = f"{SC_THRESHOLD:.2f}"
rows = []
for arch, r in RESULTS.items():
    row = {"Model": arch,
           "Val macro-F1": round(r["val"]["macro_f1"], 3),
           "Test class acc": round(r["test"]["class_acc"], 3),
           "Test macro-F1": round(r["test"]["macro_f1"], 3),
           "Test stream acc": round(r["test"]["stream_acc"], 3),
           "Stream acc (top-class lookup)": round(r["test"]["stream_acc_top_class"], 3),
           f"Special Care recall @{t20}": round(r["test"][f"sc_recall@{t20}"], 3),
           f"False alarm @{t20}": round(r["test"][f"sc_false_alarm@{t20}"], 3),
           "ECE test before → after": f"{r['ece_test_before']:.3f} → {r['ece_test_after']:.3f}",
           "Temperature": round(r["temperature"], 2)}
    if "dwsd_test" in r:
        row["Indian class acc [95% CI]"] = fmt_ci(r["dwsd_test"]["class_acc"], r["dwsd_ci"]["class_acc"])
        row["Indian stream acc [95% CI]"] = fmt_ci(r["dwsd_test"]["stream_acc"], r["dwsd_ci"]["stream_acc"])
    row.update({"Params (M)": round(r["params_m"], 2), "Size (MB)": round(r["size_mb"], 1),
                "Latency GPU (ms)": round(r.get("latency_ms_gpu", float("nan")), 1),
                "Latency CPU (ms)": round(r["latency_ms_cpu"], 1), "Train (min)": round(r["train_minutes"], 1)})
    rows.append(row)
COMPARISON = pd.DataFrame(rows).set_index("Model")
COMPARISON.to_csv(f"{RESULTS_DIR}/model_comparison.csv")
display(COMPARISON.T)

BEST = max(RESULTS, key=lambda a: RESULTS[a]["val"]["macro_f1"])
STATS["best_model"] = BEST
save_stats()
print(f"\n★ Best model on validation macro-F1: {BEST}")

In [ ]:
#@title 15 · Figures for the report and paper
COLORS = {"scratch_cnn": "#8172B3", "mobilenetv2": "#4C72B0", "efficientnetv2b0": "#55A868"}

# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for arch in RESULTS:
    log = pd.read_csv(f"{MODELS_DIR}/{arch}/training_log.csv")
    axes[0].plot(log.epoch + 1, log.val_acc, label=f"{arch} (val)", color=COLORS.get(arch))
    axes[0].plot(log.epoch + 1, log.acc, "--", alpha=0.5, color=COLORS.get(arch))
    axes[1].plot(log.epoch + 1, log.val_loss, label=arch, color=COLORS.get(arch))
axes[0].set_title("Accuracy (solid = validation, dashed = training)"); axes[0].set_xlabel("Epoch")
axes[1].set_title("Validation loss"); axes[1].set_xlabel("Epoch")
axes[0].legend(fontsize=8); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/training_curves.png", dpi=200); plt.show()

# Confusion matrices for the best model (test)
P = PRED[BEST]
yt, pt = P["y"]["test"], P["cal"]["test"]
cm = confusion_matrix(yt, pt.argmax(1), labels=range(NUM_CLASSES), normalize="true")
fig, ax = plt.subplots(figsize=(8.5, 7))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASSES, rotation=45, ha="right", fontsize=8); ax.set_yticklabels(CLASSES, fontsize=8)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if cm[i, j] >= 0.01:
            ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=6.5,
                    color="white" if cm[i, j] > 0.5 else "black")
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"{BEST}: class confusion (test, row-normalised)")
plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.savefig(f"{FIG_DIR}/confusion_classes.png", dpi=200); plt.show()

sp = stream_probs(pt)
scm = confusion_matrix(CLASS_STREAM_IDX[yt], sp.argmax(1), labels=range(len(STREAMS)))
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(scm / np.maximum(scm.sum(1, keepdims=True), 1), cmap="Greens", vmin=0, vmax=1)
ax.set_xticks(range(len(STREAMS))); ax.set_yticks(range(len(STREAMS)))
ax.set_xticklabels(STREAMS); ax.set_yticklabels(STREAMS)
for i in range(len(STREAMS)):
    for j in range(len(STREAMS)):
        ax.text(j, i, str(scm[i, j]), ha="center", va="center")
ax.set_xlabel("Predicted stream"); ax.set_ylabel("True stream"); ax.set_title(f"{BEST}: SWM stream confusion (test)")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/confusion_streams.png", dpi=200); plt.show()

# Reliability diagram before and after temperature scaling (validation)
fig, ax = plt.subplots(figsize=(4.8, 4.5))
for key, label in [("raw", "before"), ("cal", "after")]:
    pr, yy = P[key]["val"], P["y"]["val"]
    conf, corr = pr.max(1), pr.argmax(1) == yy
    edges = np.linspace(0, 1, 11)
    xs, ys = [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() >= 5:
            xs.append(conf[m].mean()); ys.append(corr[m].mean())
    ax.plot(xs, ys, "o-", label=f"{label} (ECE {ece(pr, yy):.3f})")
ax.plot([0, 1], [0, 1], "k--", lw=1); ax.set_xlabel("Predicted confidence"); ax.set_ylabel("Actual accuracy")
ax.set_title(f"{BEST}: calibration (validation)"); ax.legend()
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/calibration.png", dpi=200); plt.show()

# Special Care recall and false alarms vs threshold (all models, test)
taus = np.linspace(0.02, 0.9, 89)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for arch in RESULTS:
    pr, yy = PRED[arch]["cal"]["test"], PRED[arch]["y"]["test"]
    psc, is_sc = stream_probs(pr)[:, SC], CLASS_STREAM_IDX[yy] == SC
    axes[0].plot(taus, [(psc[is_sc] >= t).mean() for t in taus], label=arch, color=COLORS.get(arch))
    axes[1].plot(taus, [(psc[~is_sc] >= t).mean() for t in taus], label=arch, color=COLORS.get(arch))
for ax, title in zip(axes, ["Special Care recall", "False alarm rate (non-hazardous items flagged)"]):
    for r_ in SENSITIVITY_COST_RATIOS:
        ax.axvline(1 / (1 + r_), color="grey", ls=":", lw=1)
        ax.text(1 / (1 + r_), 1.01, f"{r_:g}:1", ha="center", fontsize=8, transform=ax.get_xaxis_transform())
    ax.set_xlabel("Warning threshold on P(Special Care)"); ax.set_title(title, pad=14); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/special_care_threshold.png", dpi=200); plt.show()

# Risk–coverage for the stream decision (all models, test)
fig, ax = plt.subplots(figsize=(5.5, 4.2))
for arch in RESULTS:
    pr, yy = PRED[arch]["cal"]["test"], PRED[arch]["y"]["test"]
    s = stream_probs(pr)
    cov, acc, _ = risk_coverage(s.max(1), s.argmax(1) == CLASS_STREAM_IDX[yy])
    ax.plot(cov, 1 - acc, label=arch, color=COLORS.get(arch))
ax.set_xlabel("Coverage (share of items the model answers)"); ax.set_ylabel("Error rate on answered items")
ax.set_title("Risk–coverage: SWM stream decision (test)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/risk_coverage.png", dpi=200); plt.show()

# Accuracy by data source (best model)
r = RESULTS[BEST]
names = list(r["test_by_source"]) + (["DWSD (Indian held-out)"] if "dwsd_test" in r else [])
vals = [v["class_acc"] for v in r["test_by_source"].values()] + ([r["dwsd_test"]["class_acc"]] if "dwsd_test" in r else [])
fig, ax = plt.subplots(figsize=(5.5, 3.8))
bars = ax.bar(names, vals, color=["#4C72B0", "#DD8452", "#55A868"][:len(names)])
if "dwsd_test" in r:
    lo, hi = r["dwsd_ci"]["class_acc"]
    ax.errorbar(len(names) - 1, vals[-1], yerr=[[vals[-1] - lo], [hi - vals[-1]]], color="black", capsize=5)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v + 0.01, f"{v:.2f}", ha="center")
ax.set_ylim(0, 1.08); ax.set_ylabel("Class accuracy"); ax.set_title(f"{BEST}: accuracy by image source")
plt.tight_layout(); plt.savefig(f"{FIG_DIR}/accuracy_by_source.png", dpi=200); plt.show()

report = classification_report(yt, pt.argmax(1), labels=range(NUM_CLASSES), target_names=CLASSES, zero_division=0, digits=3)
open(f"{RESULTS_DIR}/classification_report_{BEST}.txt", "w").write(report)
print(report)

## Step 4 · Grad-CAM: what the model looks at (correct and wrong predictions)

In [ ]:
#@title 16 · Grad-CAM gallery
BEST_MODEL = keras.models.load_model(f"{MODELS_DIR}/{BEST}/infer.keras", compile=False)
T_BEST = RESULTS[BEST]["temperature"]

def grad_cam(model, images, class_idx=None):
    """images: uint8 or float array (N, 224, 224, 3) in 0-255. Returns heatmaps in [0, 1] and logits."""
    pre, bb, hd = model.get_layer("preproc"), model.get_layer("backbone"), model.get_layer("head")
    x = tf.convert_to_tensor(images, tf.float32)
    with tf.GradientTape() as tape:
        fmap = bb(pre(x), training=False)
        tape.watch(fmap)
        logits = hd(fmap, training=False)
        idx = tf.argmax(logits, 1) if class_idx is None else tf.convert_to_tensor(class_idx, tf.int64)
        score = tf.gather(logits, idx, axis=1, batch_dims=1)
    grads = tape.gradient(score, fmap)
    cam = tf.nn.relu(tf.reduce_sum(tf.reduce_mean(grads, axis=(1, 2), keepdims=True) * fmap, axis=-1))
    cam = cam / (tf.reduce_max(cam, axis=(1, 2), keepdims=True) + 1e-8)
    return cam.numpy(), logits.numpy()

def overlay(img, cam, alpha=0.45):
    heat = np.array(Image.fromarray(np.uint8(cam * 255)).resize((img.shape[1], img.shape[0]), Image.BILINEAR)) / 255.0
    colored = matplotlib.colormaps["jet"](heat)[..., :3]
    return np.uint8(np.clip((1 - alpha) * img + alpha * colored * 255, 0, 255))

def load_batch(paths):
    return np.stack([load_image(p).numpy() for p in paths])

def gallery(frame, probs, title, path, per_class=2, export_prefix=None):
    pred, conf, yy = probs.argmax(1), probs.max(1), frame.y.values
    picks = []
    for c in range(NUM_CLASSES):
        m = yy == c
        if not m.any(): continue
        ok = np.nonzero(m & (pred == c))[0]; bad = np.nonzero(m & (pred != c))[0]
        picks += [(i, "correct") for i in ok[np.argsort(-conf[ok])][:per_class]]
        picks += [(i, "wrong") for i in bad[np.argsort(-conf[bad])][:per_class]]
    if not picks: return []
    cols = 4
    rows_n = math.ceil(len(picks) / cols)
    fig, axes = plt.subplots(rows_n, cols, figsize=(cols * 3, rows_n * 3.2))
    axes = np.atleast_1d(axes).ravel()
    exported = []
    for k, (i, kind) in enumerate(picks):
        img = load_batch([frame.path.iloc[i]])
        cam, _ = grad_cam(BEST_MODEL, img)
        ov = overlay(img[0], cam[0])
        axes[k].imshow(ov); axes[k].axis("off")
        axes[k].set_title(f"{'✓' if kind == 'correct' else '✗'} true: {CLASSES[yy[i]]}\npred: {CLASSES[pred[i]]} ({conf[i]:.2f})",
                          fontsize=8, color="green" if kind == "correct" else "red")
        if export_prefix:
            os.makedirs(f"{EXPORT_DIR}/gallery", exist_ok=True)
            fn = f"{export_prefix}_{CLASSES[yy[i]]}_{kind}_{k}.jpg"
            Image.fromarray(ov).save(f"{EXPORT_DIR}/gallery/{fn}", quality=92)
            exported.append(dict(file=fn, true=CLASSES[yy[i]], pred=CLASSES[pred[i]], confidence=float(conf[i]),
                                 kind=kind, source=frame.source.iloc[i]))
    for a in axes[len(picks):]: a.axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.savefig(path, dpi=160); plt.show()
    return exported

GALLERY = gallery(SPLITS["test"], PRED[BEST]["cal"]["test"], f"Grad-CAM, {BEST}: test set (✓ correct, ✗ wrong)",
                  f"{FIG_DIR}/gradcam_test.png", export_prefix="test")
if "dwsd_test" in PRED[BEST]["cal"]:
    GALLERY += gallery(SPLITS["dwsd_test"], PRED[BEST]["cal"]["dwsd_test"],
                       f"Grad-CAM, {BEST}: Indian held-out crops (DWSD)", f"{FIG_DIR}/gradcam_dwsd.png",
                       per_class=3, export_prefix="dwsd")
json.dump(GALLERY, open(f"{EXPORT_DIR}/gallery.json", "w"), indent=1)

## Step 5 · Export the best model for the Gradio app

In [ ]:
#@title 17 · Export model, settings and example images
r = RESULTS[BEST]
shutil.copy(f"{MODELS_DIR}/{BEST}/infer.keras", f"{EXPORT_DIR}/wastevision_v1.keras")
CONFIG = dict(
    version="v1", model_file="wastevision_v1.keras", architecture=BEST, img_size=IMG_SIZE,
    preprocessing="RGB -> tf.image.resize_with_pad(224, 224, antialias=True) -> round and clip to 0..255 -> float32. "
                  "No other scaling: the model normalises internally.",
    classes=CLASSES, class_display=CLASS_DISPLAY, streams=STREAMS, stream_of=STREAM_OF,
    sanitary_supported=False, temperature=r["temperature"],
    cost_ratio=COST_RATIO, special_care_threshold=SC_THRESHOLD,
    not_sure_stream_threshold=r["not_sure_stream_threshold"], not_sure_class_threshold=r["not_sure_class_threshold"],
    grad_cam="model.get_layer('preproc') -> get_layer('backbone') (last conv feature map) -> get_layer('head') (logits)",
    metrics=dict(test=r["test"], val=r["val"], dwsd_test=r.get("dwsd_test"), dwsd_ci=r.get("dwsd_ci"),
                 ece_test_after=r["ece_test_after"]),
    data=dict(totals=STATS.get("data_totals"), by_source=STATS.get("data_by_source")),
    created=datetime.datetime.now().isoformat(timespec="seconds"))
json.dump(CONFIG, open(f"{EXPORT_DIR}/config.json", "w"), indent=1)

# Example images for the app (one confident correct test image per class + Indian crops)
ex_dir = f"{EXPORT_DIR}/examples"
shutil.rmtree(ex_dir, ignore_errors=True); os.makedirs(ex_dir)
examples = []
pt, yy, te = PRED[BEST]["cal"]["test"], PRED[BEST]["y"]["test"], SPLITS["test"]
for c in range(NUM_CLASSES):
    ok = np.nonzero((yy == c) & (pt.argmax(1) == c))[0]
    if len(ok):
        i = ok[np.argsort(-pt[ok].max(1))][min(2, len(ok) - 1)]
        fn = f"{CLASSES[c]}_{te.source.iloc[i]}.jpg"
        shutil.copy(te.path.iloc[i], f"{ex_dir}/{fn}")
        examples.append(dict(file=fn, label=CLASSES[c], source=te.source.iloc[i]))
if len(SPLITS["dwsd_test"]):
    for k, row in enumerate(SPLITS["dwsd_test"].sample(min(4, len(SPLITS["dwsd_test"])), random_state=SEED).itertuples()):
        fn = f"india_{row.label}_{k}.jpg"
        shutil.copy(row.path, f"{ex_dir}/{fn}")
        examples.append(dict(file=fn, label=row.label, source="DWSD"))
json.dump(examples, open(f"{EXPORT_DIR}/examples.json", "w"), indent=1)
print(f"✓ Exported {BEST} with {len(examples)} example images to {EXPORT_DIR}")

In [ ]:
#@title 18 · Summary for the report
def md_table(df):
    cols = [str(c) for c in df.columns]
    lines = ["| " + " | ".join([df.index.name or ""] + cols) + " |", "|" + "---|" * (len(cols) + 1)]
    for idx, row in df.iterrows():
        lines.append("| " + " | ".join([str(idx)] + [str(v) for v in row.values]) + " |")
    return "\n".join(lines)

r = RESULTS[BEST]
lines = [f"# WasteVision results ({datetime.date.today()})", "",
         f"Environment: {STATS.get('environment')}", "",
         "## Data", f"Totals by split: {STATS.get('data_totals')}", f"By source: {STATS.get('data_by_source')}",
         f"Raw images found: {STATS.get('raw_counts')}", f"Excluded folders: {STATS.get('excluded_folders')}",
         f"Cleaning drops: {STATS.get('cleaning_dropped')}", f"Deduplication: {STATS.get('dedup')}",
         f"Leakage audit (held-out near-duplicates of train): {STATS.get('leakage_audit_near_duplicates_vs_train')}",
         f"DWSD: files {STATS.get('dwsd_files')}, mask formats {STATS.get('dwsd_mask_kinds')}, "
         f"mapping {STATS.get('dwsd_mask_mapping', 'index values')}, photos {STATS.get('dwsd_parent_photos')}, "
         f"groups {STATS.get('dwsd_parent_groups')}, crops by class {STATS.get('dwsd_crops_by_class')}, "
         f"skipped {STATS.get('dwsd_crops_skipped')}, zip checksum ok: {STATS.get('dwsd_zip_sha256_matches')}", "",
         "## Model comparison", md_table(COMPARISON), "",
         f"## Best model: {BEST} (selected on validation macro-F1)",
         f"Temperature: {r['temperature']:.3f}; ECE val {r['ece_val_before']:.3f} -> {r['ece_val_after']:.3f}; "
         f"ECE test {r['ece_test_before']:.3f} -> {r['ece_test_after']:.3f}",
         f"Test metrics: {json.dumps(r['test'])}",
         f"Validation metrics: {json.dumps(r['val'])}",
         f"Not-sure thresholds (from validation): stream {r['not_sure_stream_threshold']:.3f}, class {r['not_sure_class_threshold']:.3f}; "
         f"targets met: {r['not_sure_targets_met_on_val']}",
         f"Test selective stream: {r['test_selective_stream']}; selective class: {r['test_selective_class']}",
         f"Test accuracy by source: {r['test_by_source']}",
         f"Indian (DWSD) held-out: {json.dumps(r.get('dwsd_test'))}; 95% CIs: {r.get('dwsd_ci')}; groups: {r.get('dwsd_groups')}",
         "", "## Per-class report (test)", "```", report, "```",
         "", "## All models (full numbers)", "```json", json.dumps(RESULTS, indent=1), "```"]
open(f"{RESULTS_DIR}/summary.md", "w").write("\n".join(lines))
shutil.copy(f"{EXPORT_DIR}/config.json", f"{RESULTS_DIR}/export_config.json")
zip_base = f"{PROJECT_DIR}/results"
shutil.make_archive(zip_base, "zip", RESULTS_DIR)
display(Markdown(f"**Done.** Download `{zip_base}.zip` from Google Drive for the report."))

## Step 6 · Feedback retraining round (run later, after collecting feedback in the Gradio app)
The app saves each confirmed or corrected photo to `MyDrive/WasteVision/feedback/`. When about 50 corrections exist,
set `RUN_FEEDBACK_ROUND = True` below and run this cell. The new model (v2) replaces v1 in the app **only if**
stream accuracy and Special Care recall on the frozen test sets do not drop. Old-test accuracy before and after
is reported to measure forgetting.

In [ ]:
#@title 19 · Gated feedback retraining round
RUN_FEEDBACK_ROUND = False
MIN_FEEDBACK = 50
FEEDBACK_EPOCHS = 3
FEEDBACK_OVERSAMPLE = 3      # each feedback photo is shown this many times per epoch
GATE_TOLERANCE = 0.005       # allowed noise when checking "did not drop"
if os.environ.get("WV_FEEDBACK") == "1":   # local testing only
    RUN_FEEDBACK_ROUND, MIN_FEEDBACK = True, 20

def evaluate_infer_model(model, extra=None):
    out = {}
    logits = {s: model.predict(DS[s], verbose=0) for s in DS if s != "train"}
    if extra is not None and len(extra):
        logits["feedback_holdout"] = model.predict(make_ds(extra, "fb_holdout"), verbose=0)
    T = fit_temperature(logits["val"], SPLITS["val"].y.values)
    for s, l in logits.items():
        y_ = extra.y.values if s == "feedback_holdout" else SPLITS[s].y.values
        out[s] = metrics(softmax(l / T, 1), y_)
    out["temperature"] = T
    return out

if RUN_FEEDBACK_ROUND:
    fb = pd.read_csv(f"{FEEDBACK_DIR}/labels.csv")
    fb = fb[fb.human_label.isin(CLASSES)].copy()
    print(f"Feedback photos with a valid label: {len(fb)}")
    assert len(fb) >= MIN_FEEDBACK, f"Need at least {MIN_FEEDBACK} labelled feedback photos (have {len(fb)})."
    fb["source"], fb["label"] = "Feedback", fb.human_label
    fb["id"] = [stable_id("FB", f) for f in fb.file]
    fb["src_path"] = FEEDBACK_DIR + "/images/" + fb.file
    fb["bbox"], fb["group"] = None, fb.id
    fb = normalize_all(fb[["id", "source", "src_path", "label", "bbox", "group"]].assign(orig_class=fb.label, rel=fb.file))
    fb["y"] = fb.label.map(CLASS_TO_IDX)
    fb["path"] = NORM_DIR + "/" + fb.norm_rel
    fb_train, fb_hold = (strat_split(fb.assign(strat=fb.label), 0.2) if len(fb) >= 20 else (fb, fb.iloc[:0]))

    cfg = json.load(open(f"{EXPORT_DIR}/config.json"))
    v1 = keras.models.load_model(f"{EXPORT_DIR}/{cfg['model_file']}", compile=False)
    before = evaluate_infer_model(v1, fb_hold)

    v2 = keras.models.load_model(f"{EXPORT_DIR}/{cfg['model_file']}", compile=False)
    if cfg["architecture"] != "scratch_cnn":
        unfreeze_top(v2.get_layer("backbone"), UNFREEZE_LAST_LAYERS)
    inp = keras.Input((IMG_SIZE, IMG_SIZE, 3))
    train_v2 = keras.Model(inp, v2(make_augmenter()(inp)))
    compile_model(train_v2, LR_FINETUNE)
    combined = pd.concat([SPLITS["train"]] + [fb_train] * FEEDBACK_OVERSAMPLE, ignore_index=True)
    combined["id"] = combined.id + "_" + combined.groupby("id").cumcount().astype(str)
    fit(train_v2, make_ds(combined, "train_feedback", training=True), FEEDBACK_EPOCHS, 0, 2,
        f"{MODELS_DIR}/feedback_round_log.csv")
    after = evaluate_infer_model(v2, fb_hold)

    t = f"{SC_THRESHOLD:.2f}"
    gate = (after["test"]["stream_acc"] >= before["test"]["stream_acc"] - GATE_TOLERANCE and
            after["test"][f"sc_recall@{t}"] >= before["test"][f"sc_recall@{t}"] - GATE_TOLERANCE)
    comp = pd.DataFrame({s: {"v1": before[s]["class_acc"], "v2": after[s]["class_acc"]}
                         for s in before if s != "temperature"}).T
    comp["change"] = comp.v2 - comp.v1
    display(comp)
    print(f"Old test set: stream acc {before['test']['stream_acc']:.3f} -> {after['test']['stream_acc']:.3f}; "
          f"Special Care recall {before['test'][f'sc_recall@{t}']:.3f} -> {after['test'][f'sc_recall@{t}']:.3f}")
    json.dump(dict(before=before, after=after, accepted=gate, n_feedback=len(fb), n_feedback_train=len(fb_train),
                   n_feedback_holdout=len(fb_hold)), open(f"{RESULTS_DIR}/feedback_round.json", "w"), indent=1)
    if gate:
        v2.save(f"{EXPORT_DIR}/wastevision_v2.keras")
        cfg.update(version="v2", model_file="wastevision_v2.keras", temperature=after["temperature"],
                   metrics=dict(test=after["test"], val=after["val"], dwsd_test=after.get("dwsd_test")),
                   updated=datetime.datetime.now().isoformat(timespec="seconds"))
        json.dump(cfg, open(f"{EXPORT_DIR}/config.json", "w"), indent=1)
        print("✓ Gate passed: v2 is now the app's model (v1 kept as a backup).")
    else:
        print("✗ Gate failed: v2 would make the frozen test results worse, so v1 stays in the app.")
    shutil.make_archive(f"{PROJECT_DIR}/results", "zip", RESULTS_DIR)
else:
    print("Feedback round not requested. Set RUN_FEEDBACK_ROUND = True after collecting feedback in the app.")